# Get a local FITS sample (Local case)

Run these commands in a terminal from the repository root. The first `rsync` lists the remote files and selects five; the second copies those five locally. SSH will prompt for your password for each call.

```sh
mkdir -p data/mojave_fits_sample
REMOTE='nuradio:/cephfs/projects/RADIO/mojave_archive/intensities_fits/'
rsync -av -e ssh "$REMOTE" data/mojave_fits_sample/
```

control C to cancel the download 

If on the server just change the fits_dir and uv_dir

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").is_file():
    repo_root = repo_root.parent
fits_dir = repo_root / "data" / "mojave_fits_sample"
uv_dir = repo_root / "data" / "mojave_uv_sample"
fits_files = sorted(fits_dir.glob("*.fits.gz"))
uv_files = sorted(uv_dir.glob("*.uvf"))

print(f"FITS directory: {fits_dir}")
print(f"Files found: {len(fits_files)}")
for index, path in enumerate(fits_files):
    print(f"{index}: {path.name}")
if not fits_files:
    print("No local FITS files found. Run the terminal commands in the cell above first.")

In [ ]:
if not fits_files:
    raise FileNotFoundError(f"No FITS files found in {fits_dir}")

sample_path = fits_files[1]
print(f"Inspecting: {sample_path.name}")

with fits.open(sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[0].header)
    display(hdul[0].data.shape)

In [ ]:
with fits.open(sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[1].header)
    display(hdul[1].data)

In [ ]:
uv_sample_path = uv_files[0]
print(f"Inspecting: {uv_sample_path.name}")

with fits.open(uv_sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[0].header)

In [ ]:
with fits.open(uv_sample_path, memmap=False) as hdul:
    display(hdul[0].data)

In [ ]:
with fits.open(uv_sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[1].header)

In [ ]:
with fits.open(uv_sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[2].header)

In [ ]:
with fits.open(uv_sample_path, memmap=False) as hdul:
    hdul.info()
    print("\nPrimary HDU header:")
    display(hdul[3].header)

In [ ]:
import pandas as pd
from IPython.display import display

header_rows = []
for path in fits_files:
    with fits.open(path, memmap=False) as hdul:
        for hdu_index, hdu in enumerate(hdul):
            row = {
                "filename": path.name,
                "hdu_index": hdu_index,
                "extname": hdu.header.get("EXTNAME", "PRIMARY" if hdu_index == 0 else None),
            }
            row.update(
                (key, hdu.header[key])
                for key in hdu.header
                if key not in {"COMMENT", "HISTORY"}
            )
            header_rows.append(row)

headers_df = pd.DataFrame(header_rows)
axis_columns = [
    "filename", "hdu_index", "extname", "NAXIS3", "CTYPE3",
    "CUNIT3", "CRVAL3", "CRPIX3", "CDELT3",
]
display(headers_df.reindex(columns=axis_columns))

if "CRVAL3" in headers_df.columns:
    print("Distinct CRVAL3 values per HDU index:")
    display(headers_df.groupby("hdu_index")["CRVAL3"].nunique().rename("distinct_CRVAL3_count"))

In [ ]:
# Summarize observation metadata and CLEAN components across the local sample.
import pandas as pd
from IPython.display import display

if not fits_files:
    raise FileNotFoundError(f"No FITS files found in {fits_dir}")

metadata_fields = [
    "OBJECT", "DATE-OBS", "CTYPE3", "CRVAL3", "CUNIT3",
    "CDELT1", "CDELT2", "CUNIT1", "CUNIT2",
    "BMAJ", "BMIN", "BPA", "NOISE", "BUNIT",
]
metadata_records = []
component_records = []
component_examples = None
component_type_counts = {}

for path in fits_files:
    with fits.open(path, memmap=False) as hdul:
        metadata_records.append({
            "filename": path.name,
            **{key: hdul[0].header.get(key) for key in metadata_fields},
        })
        cc_hdus = [
            hdu for hdu in hdul[1:]
            if str(hdu.header.get("EXTNAME", "")).strip() == "AIPS CC"
        ]
        if not cc_hdus:
            component_records.append({
                "filename": path.name, "status": "No AIPS CC extension",
            })
        for hdu in cc_hdus:
            # Convert FITS byte order before using pandas numeric operations.
            component_array = np.array(hdu.data)
            components = pd.DataFrame.from_records(
                component_array.astype(component_array.dtype.newbyteorder("="))
            )
            components.columns = components.columns.str.strip()
            row = {
                "filename": path.name,
                "extver": hdu.header.get("EXTVER"),
                "status": "Present",
                "component_count": len(components),
                "model_flux_sum_jy": (
                    components["FLUX"].sum(skipna=False)
                    if "FLUX" in components else float("nan")
                ),
                "type_obj_values": None,
            }
            if "TYPE OBJ" in components:
                row["type_obj_values"] = sorted(
                    components["TYPE OBJ"].dropna().unique().tolist()
                )
                for value, count in components["TYPE OBJ"].value_counts().items():
                    component_type_counts[value] = (
                        component_type_counts.get(value, 0) + int(count)
                    )
            component_records.append(row)
            if component_examples is None and not components.empty:
                component_examples = components.head(5).copy()
                component_examples.insert(0, "extver", hdu.header.get("EXTVER"))
                component_examples.insert(0, "filename", path.name)

metadata_df = pd.DataFrame(metadata_records)
summary_rows = []
for key in metadata_fields:
    values = metadata_df[key]
    present = values.dropna()
    numeric = pd.to_numeric(present, errors="coerce").dropna()
    summary_rows.append({
        "field": key,
        "missing_files": int(values.isna().sum()),
        "distinct_values": int(present.nunique()),
        "numeric_min": numeric.min(),
        "numeric_max": numeric.max(),
        "example_values": present.unique()[:5].tolist(),
    })

print(f"Primary-header summary for {len(metadata_df)} files (raw header units):")
print("Missing CUNIT values are reported explicitly; no unit conversion is applied.")
display(pd.DataFrame(summary_rows).set_index("field"))
print("CLEAN-component tables (model flux refers to the full table, before cropping):")
display(pd.DataFrame(component_records))
print("Observed TYPE OBJ values and component counts across all CC tables:")
display(pd.DataFrame(
    sorted(component_type_counts.items()), columns=["TYPE OBJ", "component_count"]
))
if component_examples is not None:
    print("First five component rows from the first non-empty CC table (table units):")
    display(component_examples)
